# 公开 TabPFN 预测复现

四路TabPFN概率由 **TaeyanG4** 公开，融合方法参考 **kalidenaiarys** 的Notebook。
本项目重新拟合元模型，没有自行训练TabPFN。来源与许可见 `reference/ATTRIBUTION.md`。

两份实际提交成绩：基线 **0.82604**，融合 **0.82534**。本Notebook重新生成对应文件；不自动向Kaggle提交。

In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p/'spaceship/train.py').exists())
sys.path.insert(0,str(ROOT))

## 准备公开概率
首次运行会下载已经审阅的公开概率包，逐个检查固定校验值，保存至Git忽略的 `.cache/tabpfn/`。
下载器只解析数据字面量，不执行远程Notebook代码。

In [2]:
if not (ROOT/'.cache/tabpfn/provenance.json').exists():
    result=subprocess.run([sys.executable,'-m','spaceship.assets'],cwd=ROOT,
        check=True,capture_output=True,text=True)
    print(result.stdout)
else:
    print('已存在本地公开概率缓存，复现脚本会重新检查其校验值。')

已存在本地公开概率缓存，复现脚本会重新检查其校验值。


## 重新拟合元模型
基线用四路概率的logit拟合逻辑回归。融合方案等权平均样条元模型和加入CatBoost的逻辑回归概率。
基础概率固定，元模型开发OOF不是完全嵌套重训，也不能与此前随机分层分数直接横比。

In [3]:
result=subprocess.run([sys.executable,'-m','spaceship.replay'],cwd=ROOT,
    check=True,capture_output=True,text=True)
print(result.stdout)

tabpfn_baseline fold 1: 82.4612%
tabpfn_baseline fold 2: 84.4738%
tabpfn_baseline fold 3: 83.9563%
tabpfn_baseline fold 4: 82.2209%
tabpfn_baseline fold 5: 82.4511%
tabpfn_spline fold 1: 82.4037%
tabpfn_spline fold 2: 84.5313%
tabpfn_spline fold 3: 84.0138%
tabpfn_spline fold 4: 82.3936%
tabpfn_spline fold 5: 82.6237%
tabpfn_cat_logistic fold 1: 82.6912%
tabpfn_cat_logistic fold 2: 84.4738%
tabpfn_cat_logistic fold 3: 83.7263%
tabpfn_cat_logistic fold 4: 82.5662%
tabpfn_cat_logistic fold 5: 82.6237%
tabpfn_baseline {'file': 'submission_tabpfn_baseline.csv', 'sha256': 'cba1b98a2a8ad88ed7a5d00113963bb64dfcf9ae9c37bd45bf700f2817953431', 'expected_source_sha256': 'cba1b98a2a8ad88ed7a5d00113963bb64dfcf9ae9c37bd45bf700f2817953431', 'matches_source_exactly': True, 'source_reported_public_accuracy': 0.82604, 'different_predictions_vs_v2_in_original_project': 296}
tabpfn_catboost_blend {'file': 'submission_tabpfn_catboost_blend.csv', 'sha256': 'd171afa54c18e5cab46dd570d0283c193bc616d10ae186a82a

## 复核结果与两个文件的差异

In [4]:
result=subprocess.run([sys.executable,'-m','spaceship.verify'],cwd=ROOT,
    check=True,capture_output=True,text=True)
print(result.stdout)
display(pd.read_csv(ROOT/'runs/replay/model_comparison.csv'))
changes=pd.read_csv(ROOT/'runs/replay/baseline_vs_blend_changes.csv',dtype={'PassengerId':str})
assert len(changes)==7
display(changes)

Verified both source submission hashes, 4,277 IDs, boolean labels, OOF scores and 7 differing predictions.



,model,development_meta_oof_accuracy
0,tabpfn_baseline,0.831128
1,tabpfn_spline,0.831934
2,tabpfn_cat_logistic,0.832164
3,tabpfn_catboost_blend,0.832164


,PassengerId,baseline_probability,blend_probability
0,0752_01,0.499306,0.500316
1,2603_01,0.502579,0.499266
2,3675_01,0.502563,0.498800
3,6287_01,0.486952,0.501439
4,6314_01,0.497643,0.505921
5,8800_03,0.498273,0.501526
6,9050_01,0.499010,0.501104


## 提交文件

- `runs/replay/submission_tabpfn_baseline.csv`
- `runs/replay/submission_tabpfn_catboost_blend.csv`

两份各4277行，分别与来源提交SHA256一致。代码重跑不代表新的排行榜评估。
最佳成绩来自公开预测复现，署名与学习总结应保留这一点。